# Part 2 — Methane liquid–vapour coexistence in GROMACS
**Dr Valentina Erastova**

Same *idea* as Part 1: **liquid–vapour coexistence** of a simple fluid.  
Here the fluid is **methane** (united-atom LJ site — TraPPE-like), the main component of natural gas and of interest for **methane hydrates** (“combustible ice”) in subsurface / offshore settings.  
New layer: GROMACS files, especially the **`.mdp`**.

<div style="background-color:rgba(47,128,237,0.08);padding:10px 14px;border-left:4px solid #2F80ED;margin:10px 0;border-radius:4px;">
Before this notebook: finish setup in <b><a href="../STUDENT_GUIDE.md">STUDENT_GUIDE.md</a></b> (Python packages + <code>gmx --version</code>).<br>
Copy GROMACS commands into a <b>terminal</b> from the <code>part2_gromacs</code> directory. Long coexistence runs are <b>precomputed</b> — you edit inputs and interpret outputs.
</div>

**Came from Part 1?** [LJ coexistence applet](../part1_interactive_md/LJ_coexistence_applet.ipynb)  
Part 1 was a toy LJ fluid; Part 2 keeps the coexistence + MDP habits on a **reservoir-relevant** molecule.

**Temperatures (kelvin):** stay below methane’s critical temperature (~190 K).

| T (K) | MDP files |
|-------|-----------|
| 110 | `nvt_T110.mdp`, `prod_T110.mdp` |
| 120 | `nvt_T120.mdp`, `prod_T120.mdp` |
| 130 | `nvt_T130.mdp`, `prod_T130.mdp` |
| 140 | `nvt_T140.mdp`, `prod_T140.mdp` |
| 150 | `nvt_T150.mdp`, `prod_T150.mdp` |
| 170 | `nvt_T170.mdp`, `prod_T170.mdp` |


## 1. Files, file types, and the slab geometry

Work from the `part2_gromacs` directory:

```bash
cd part2_gromacs
head structures/slab_start.gro
tail -1 structures/slab_start.gro
cat inputs/topol.top
cat FF/ch4.itp
cat FF/lj_methane.itp
```

### GROMACS file types

| File type | Role |
|-----------|------|
| `.gro` | Coordinates + box dimensions |
| `.itp` | Reusable molecule / force-field piece |
| `.top` | System topology: includes FF + molecule counts |
| `.mdp` | How to run (dt, T, thermostat, cutoffs, …) — input to `grompp` |
| `.tpr` | Binary run input produced by `grompp` |
| `.cpt` | Checkpoint (velocities / state) to continue a run |
| `.edr` | Energies, T, P, … |
| `.log` | Settings, warnings, performance |
| `.xtc` / `.trr` | Trajectory |

### The core pipeline (memorise this)

Every dynamics stage repeats the same pattern:

```text
  coordinates (.gro)  +  topology (.top)  +  MDP (.mdp)
                         |
                         v
                    gmx grompp
                         |
                         v
                       .tpr
                         |
                         v
                    gmx mdrun
                         |
                         v
              .gro  .xtc  .edr  .log  .cpt  …
```

| Flag | Meaning |
|------|---------|
| `grompp -f` | `.mdp` |
| `       -c` | input coordinates (`.gro`) |
| `       -p` | topology (`.top`) |
| `       -t` | optional checkpoint (`.cpt`) to continue |
| `       -o` | output `.tpr` |
| `mdrun -deffnm` | common prefix for the `.tpr` and all outputs |

### Paths in *this* project

| Path | Role |
|------|------|
| `structures/slab_start.gro` | Starting coordinates + box (elongated in **z**) |
| `FF/lj_methane.itp` | UA methane force field (TraPPE-like) |
| `FF/ch4.itp` | One CH4 site |
| `inputs/topol.top` | System: 500 CH4 |
| `MDP/*.mdp` | EM / NVT / production controls |
| `precomputed/T_*/` | Finished production runs (shipped) |
| `em/`, `nvt/`, `prod/`, `analysis/` | **Created when you run** — scratch outputs |

**Why a slab?** Liquid in the middle, vapour at the ends → one box contains **both** coexisting phases.  
**Why NVT (no barostat)?** Fixed volume lets the interface set the orthobaric densities.  
**Why united-atom?** One LJ site per CH4 is standard for alkane phase equilibria (e.g. TraPPE). Full atomistic CH4 adds C–H detail; validation *spirit* is the same.


## 2. Read and write the MDP (main hands-on)

Open the template and one prepared file:

```bash
less MDP/prod_template.mdp
less MDP/prod_T130.mdp
```

Find and discuss:

| Part 1 idea | MDP entry |
|-------------|-----------|
| Timestep | `dt` |
| Length of run | `nsteps` (time = dt × nsteps) |
| Temperature | `ref_t` (**kelvin** here) |
| Thermostat | `tcoupl`, `tau_t` |
| No pressure coupling | `pcoupl = no` |
| PBC | `pbc = xyz` |

### Task
Create a new production MDP for an extra temperature of your choice, e.g. T = 140 K:

```bash
cp MDP/prod_template.mdp MDP/prod_T140.mdp
# edit ref_t to 140
nano MDP/prod_T140.mdp    # or vim / VS Code
```

You will **not** wait for this run in the room; the point is to practise the edit.


## 3. GROMACS pipeline — EM → NVT → production

Staged workflow for this slab (and for later GROMACS sessions in the school):

```text
  slab_start.gro
        |
        v
   [1] Energy minimisation (EM)     ← remove bad contacts
        |
        v
   [2] NVT equilibration            ← settle at chosen T (here 130 K)
        |
        v
   [3] Production (NVT slab)        ← sample coexistence  (use precomputed/ in class)
        |
        v
   [4] Analysis                     ← gmx density, gmx energy, …
```

Scratch folders (not shipped — you create them):

```bash
mkdir -p em nvt prod analysis
```

---

### Stage 1 — Energy minimisation

**Purpose:** relax overlaps / bad contacts before dynamics.  
**Inputs:** `MDP/em.mdp` + `structures/slab_start.gro` + `inputs/topol.top`  
**Outputs:** `em/em.tpr` → `em/em.gro`, `em/em.log`, …

```bash
gmx grompp -f MDP/em.mdp -c structures/slab_start.gro -p inputs/topol.top -o em/em.tpr
gmx mdrun  -deffnm em/em -nt 4
```

---

### Stage 2 — NVT equilibration (demo T = 130 K)

**Purpose:** bring the slab to the target temperature; generate velocities.  
**Inputs:** `MDP/nvt_T130.mdp` + `em/em.gro` + `inputs/topol.top`  
**Outputs:** `nvt/nvt_T130.tpr` → `nvt/nvt_T130.gro`, `.cpt`, `.edr`, …

```bash
gmx grompp -f MDP/nvt_T130.mdp -c em/em.gro -p inputs/topol.top -o nvt/nvt_T130.tpr
gmx mdrun  -deffnm nvt/nvt_T130 -nt 4
```

---

### Stage 3 — Production (optional in the room — do not wait)

**Purpose:** sample liquid–vapour coexistence at fixed T (still **NVT**, `pcoupl = no`).  
**Inputs:** `MDP/prod_T130.mdp` + `nvt/nvt_T130.gro` + checkpoint + `inputs/topol.top`  
**Outputs:** `prod/prod_T130.xtc`, `.edr`, `.gro`, …

```bash
gmx grompp -f MDP/prod_T130.mdp \
  -c nvt/nvt_T130.gro -t nvt/nvt_T130.cpt \
  -p inputs/topol.top -o prod/prod_T130.tpr
gmx mdrun  -deffnm prod/prod_T130 -nt 4
```

Note the **`-t`** checkpoint: production **continues** from equilibrated velocities (same habit when chaining equilibration → production).

---

### Stage 4 — Analysis (preview; full curve in §4)

```bash
echo 0 | gmx density -f precomputed/T_130/prod.xtc -s precomputed/T_130/prod.tpr -d Z -o analysis/density_T130.xvg
echo Temperature | gmx energy -f precomputed/T_130/prod.edr -o analysis/T_T130.xvg
```

---

**Remember:** `.mdp` + `.gro` + `.top` → **`grompp`** → `.tpr` → **`mdrun`**.  
For the coexistence curve in class, use shipped **`precomputed/`** (next section), not a full live production.


## 4. Precomputed coexistence curve

Folders `precomputed/T_110`, `T_120`, `T_130`, `T_140`, `T_150`, `T_170` contain finished production outputs:
`prod.xtc`, `prod.edr`, `prod.tpr`, `prod.gro`, `prod.log`, and a density profile `density_z.xvg`.

Density units in the `.xvg` files: **kg m⁻³**. *z* is in **nm** along the long box axis.

### Step A — look at the density profiles (ρ vs *z*)

**Optional**: regenerate one profile with GROMACS

```bash
mkdir -p analysis
echo 0 | gmx density -f precomputed/T_130/prod.xtc -s precomputed/T_130/prod.tpr -d Z -o analysis/density_T130.xvg
```

Visualise the plot:

```bash
xmgrace analysis/density_T130.xvg
```

**Preferred (all temperatures on one plot):** from `part2_gromacs` run

```bash
xmgrace precomputed/T_*/density_z.xvg -legend load
```

That overlays every shipped `T_*` profile with a legend. Compare how the liquid (high-ρ region) and vapour (low-ρ ends) change with temperature.

_No xmgrace? Use the Python cell below instead (same profiles + coexistence sketch + $T_c$)._

You should see:

- **outer plateaus** (low ρ) → vapour  
- **high-ρ band** (liquid slab; may sit off-centre) → liquid  
- steep rises/falls → the two interfaces  

**Clear two-phase profiles:** 110, 120, 130, 140 K.  
**Nearly flat / one fluid:** 150–170 K (closer to methane $T_c \approx 190\,\mathrm{K}$, and finite-size loss of the liquid slab). Use **110–140 K** for $\rho_{\mathrm{liq}}$/$\rho_{\mathrm{vap}}$ and the $T_c$ fit.


### Step B — estimate $\rho_{\mathrm{liq}}$ and $\rho_{\mathrm{vap}}$ by eye

For each T in {110, 120, 130, 140} K (and glance at 150, 170):

1. Plot `precomputed/T_XXX/density_z.xvg` (Step A / cell below).  
2. **$\rho_{\mathrm{vap}}$** ≈ average of the flat region near the **ends**.  
3. **$\rho_{\mathrm{liq}}$** ≈ **peak** of the liquid (high-ρ) plateau — read the top of the flat/high region, not a mid-box average if the slab is off-centre.  
4. Fill a table (rough ±10–20 kg m⁻³ is fine in class):

| T (K) | $\rho_{\mathrm{liq}}$ (kg m$^{-3}$) | $\rho_{\mathrm{vap}}$ (kg m$^{-3}$) |
|-------|----------------|----------------|
| 110 | | |
| 120 | | |
| 130 | | |
| 140 | | |
| 150 | (optional — likely unclear) | |
| 170 | (optional — likely one phase) | |


### Step C — plot the coexistence curve ρ(T)

Put **T on the x-axis** and **density on the y-axis**, with two series:

- $\rho_{\mathrm{liq}}(T)$ — usually the upper branch  
- $\rho_{\mathrm{vap}}(T)$ — usually the lower branch  

That is the liquid–vapour envelope (same coexistence idea as Part 1, now as numbers).  
The Python cell plots this from your estimates (edit the example numbers). Sketching by hand is also fine for discussion.


### Step D — theory: coexistence curve and estimating $T_c$

#### Why this curve matters

Below the critical temperature $T_c$, a pure fluid can exist as **liquid and vapour in equilibrium**. At each such T:

- liquid density $\rho_{\mathrm{liq}}(T)$ (high)
- vapour density $\rho_{\mathrm{vap}}(T)$ (low)
- they share the same vapour pressure

Plotting $\rho_{\mathrm{liq}}(T)$ and $\rho_{\mathrm{vap}}(T)$ gives the **liquid–vapour coexistence envelope**. As $T \to T_c$, the two densities approach each other and the interface disappears. Above $T_c$ there is only one fluid phase.

In reservoir / oil modelling the same idea appears as a **phase envelope** (EOS / PVT): you need to know whether one or two phases exist and what their densities are. In MD, matching this envelope is a standard **force-field validation** test (Sofia’s loop) — before trusting the model for interfaces or more complex chemistry.

#### What we measure in the slab

From $\rho(z)$:

- **outer plateaus** → $\rho_{\mathrm{vap}}$
- **peak of the high-ρ plateau** → $\rho_{\mathrm{liq}}$

Use only T where **two clear plateaus** remain: **110, 120, 130, 140 K**.  
At 150–170 K this small box is nearly one fluid — those points are **not** coexistence densities for a $T_c$ fit.

#### Estimating $T_c$: scaling of $\Delta\rho$

The branches meet at

$$\rho_{\mathrm{liq}}(T_c)=\rho_{\mathrm{vap}}(T_c)=\rho_c.$$

Close to criticality,

$$\Delta\rho(T)=\rho_{\mathrm{liq}}-\rho_{\mathrm{vap}}\;\propto\;(T_c-T)^{\beta},\qquad \beta\approx 0.325.$$

That $\beta$ is the **3D Ising** critical exponent for the order parameter (universality class of ordinary fluids) — the same fixed value used in TraPPE-style vapour–liquid coexistence curve (VLCC) / $T_c$ fits. It is **not** fitted from our slab, and not a TraPPE force-field parameter.

Linearise $[\Delta\rho]^{1/\beta}$ vs $T$ and extrapolate to zero → $T_c$ (cells below). Use the rectilinear diameter for $\rho_c$.

**TraPPE methane target:** $T_c \approx 190\,\mathrm{K}$. Proper TraPPE validation (Gibbs-ensemble MC, large systems, points near $T_c$) is typically **on target or slightly high** vs experiment — TraPPE was built for phase equilibria.

A **classroom slab** like ours often **underestimates** $T_c$. That is **not** TraPPE failing: small NVT slab, short runs, only 110–140 K (far below $T_c$), and finite-size loss of the liquid film all pull the scaling fit down. Compare the *shape* of the envelope to TraPPE ethane on http://trappe.oit.umn.edu/validation/ (one C more); do not expect our extrapolated $T_c$ to hit 190 K.

### Optional health check

```bash
echo Temperature | gmx energy -f precomputed/T_130/prod.edr -o analysis/T_T130.xvg
```


### Plot A — all density profiles on one figure

Run the cell below to overlay `density_z.xvg` for every shipped temperature.  
Read **$\rho_{\mathrm{liq}}$** from the **peak** of the liquid band and **$\rho_{\mathrm{vap}}$** from the outer low plateau.


In [ ]:
# A — all T on one ρ(z) plot  (run from part2_gromacs)
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

def load_xvg(path):
    z, rho = [], []
    for line in Path(path).read_text().splitlines():
        if not line.strip() or line[0] in "#@":
            continue
        a, b = line.split()[:2]
        z.append(float(a))
        rho.append(float(b))
    return np.array(z), np.array(rho)

temps = [110, 120, 130, 140, 150, 170]
base = Path("precomputed")

plt.figure(figsize=(8, 5))
for T in temps:
    path = base / f"T_{T}" / "density_z.xvg"
    if not path.is_file():
        print(f"missing {path}")
        continue
    z, rho = load_xvg(path)
    plt.plot(z, rho, label=f"{T} K")
plt.xlabel("z (nm)")
plt.ylabel(r"$\rho$ (kg m$^{-3}$)")
plt.title("Methane slab density profiles (all T)")
plt.legend(title="T")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Plot B — coexistence densities $\rho_{\mathrm{liq}}(T)$ and $\rho_{\mathrm{vap}}(T)$

Fill / edit the dictionaries from Plot A (**peak** = liquid, outer plateau = vapour).  
Use `None` where there are not two clear phases (typically 150–170 K).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
# B — your plateau estimates → coexistence points
# ρ_liq = PEAK of the liquid band on Plot A
rho_liq = {110: 400, 120: 375, 130: 355, 140: 310, 150: None, 170: None}  # kg m⁻³
rho_vap = {110: 3,   120: 5,   130: 10,  140: 20,  150: None, 170: None}

T_coex, liq, vap = [], [], []
for T in sorted(rho_liq):
    if rho_liq[T] is not None and rho_vap[T] is not None:
        T_coex.append(float(T))
        liq.append(float(rho_liq[T]))
        vap.append(float(rho_vap[T]))
T_coex = np.array(T_coex)
liq = np.array(liq)
vap = np.array(vap)

print("Coexistence points:")
for T, rl, rv in zip(T_coex, liq, vap):
    print(f"  T = {T:.0f} K :  ρ_liq = {rl:.0f} ,  ρ_vap = {rv:.0f}  kg m⁻³")

plt.figure(figsize=(6, 4))
plt.plot(T_coex, liq, "o-", label=r"$\rho_{\mathrm{liq}}$ (peaks)")
plt.plot(T_coex, vap, "s-", label=r"$\rho_{\mathrm{vap}}$ (outer)")
plt.xlabel("T (K)")
plt.ylabel(r"$\rho$ (kg m$^{-3}$)")
plt.title("Liquid–vapour coexistence densities")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Calculate $T_c$ — scaling fit

Near the critical point the gap closes as a **power law**:

$$
\Delta\rho(T) \equiv \rho_{\mathrm{liq}}(T)-\rho_{\mathrm{vap}}(T)
\;\propto\;
(T_c-T)^{\beta},
\qquad \beta \approx 0.325
$$

$\beta \approx 0.325$ is the **3D Ising** exponent (ordinary fluids). TraPPE-style analyses fix the same value when extrapolating $T_c$ — it is not pulled from our data, and not a force-field parameter.

**Classroom method:** linearise

$$
\left[\Delta\rho(T)\right]^{1/\beta} = C\,(T_c - T)
$$

and find where the fit hits **zero** → $T_c$. The rectilinear diameter gives $\rho_c$:

$$
\frac{\rho_{\mathrm{liq}}+\rho_{\mathrm{vap}}}{2} \approx \rho_c + A(T_c-T)
$$

**Expect a low classroom $T_c$** (often ~160 K with the example numbers) vs TraPPE / literature **≈ 190 K**. TraPPE itself is usually bang-on or a bit high; our underestimate comes from the crude slab protocol (tiny NVT box, short runs, T window far from criticality) — not from the force field being “too cold”.


In [ ]:
# C — scaling estimate of Tc from Δρ ∝ (Tc − T)^β
# Needs Plot B cell run first (T_coex, liq, vap).
import numpy as np

if len(T_coex) < 2:
    raise SystemExit("Need at least two coexistence points (edit rho_liq / rho_vap in Plot B).")

beta = 0.325
drho = liq - vap
if np.any(drho <= 0):
    raise SystemExit("Δρ must be positive at each fitted T.")

# Linearise: y = (Δρ)^(1/β) = C (Tc − T) = (−C) T + C Tc
y = drho ** (1.0 / beta)
m, c = np.polyfit(T_coex, y, 1)
C = -m
Tc = -c / m

# Rectilinear diameter → ρ_c
rho_mean = 0.5 * (liq + vap)
A_slope, A_int = np.polyfit(T_coex, rho_mean, 1)  # slope = -A
A = -A_slope
rho_c = A_int - A * Tc

print(f"β = {beta}")
print("Coexistence points:")
for T, rl, rv, d in zip(T_coex, liq, vap, drho):
    print(f"  T = {T:.0f} K :  ρ_liq = {rl:.0f} ,  ρ_vap = {rv:.0f} ,  Δρ = {d:.0f}  kg m⁻³")

print(f"\nLinearised fit: [Δρ]^(1/β) = {m:.4g} × T + {c:.4g}")
print(f"\n>>> Classroom Tc ≈ {Tc:.1f} K")
print(f">>> Classroom ρ_c ≈ {rho_c:.1f} kg m⁻³")
print(">>> TraPPE / literature methane Tc ≈ 190 K")
print(">>> (Low classroom Tc is expected here: small NVT slab ≠ TraPPE Gibbs-ensemble validation.)")


### Plot D — TraPPE-style VLCC (vapour–liquid coexistence curve)

**VLCC** = vapour–liquid coexistence curve. **Density on $x$, temperature on $y$** (vapour left, liquid right, critical point at the top) — same layout as TraPPE validation plots. Ethane example (one C more): http://trappe.oit.umn.edu/validation/

Markers: your estimates. Curves: scaling reconstruction from cell C. Star: classroom $(T_c,\rho_c)$. Dashed / dotted horizontals: classroom $T_c$ vs literature methane ≈ 190 K.


In [ ]:
# D — TraPPE-style VLCC = vapour–liquid coexistence curve (ρ on x, T on y)
# Needs Plot B and calculation cell C run first.
import numpy as np
import matplotlib.pyplot as plt

T_min = float(T_coex.min())
T_line = np.linspace(T_min, Tc, 300)
dT = np.clip(Tc - T_line, 0, None)
drho_fit = (C * dT) ** beta
rho_mean_fit = rho_c + A * dT
rho_liq_fit = rho_mean_fit + 0.5 * drho_fit
rho_vap_fit = rho_mean_fit - 0.5 * drho_fit

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot(vap, T_coex, "s", ms=8, color="C1", label=r"$\rho_{\mathrm{vap}}$ (your plateaus)")
ax.plot(liq, T_coex, "o", ms=8, color="C0", label=r"$\rho_{\mathrm{liq}}$ (your peaks)")
ax.plot(rho_vap_fit, T_line, "-", color="C1", alpha=0.9, label=r"scaling $\rho_{\mathrm{vap}}$")
ax.plot(rho_liq_fit, T_line, "-", color="C0", alpha=0.9, label=r"scaling $\rho_{\mathrm{liq}}$")
ax.plot([rho_vap_fit[-1], rho_liq_fit[-1]], [Tc, Tc], "-", color="0.35", lw=1.2)
ax.plot([rho_c], [Tc], "k*", ms=14, label=fr"classroom $(T_c,\rho_c)\approx({Tc:.0f}\,\mathrm{{K}},\,{rho_c:.0f})$")
ax.axhline(Tc, color="0.35", ls="--", alpha=0.8, label=fr"classroom $T_c\approx{Tc:.0f}$ K")
ax.axhline(190, color="C3", ls=":", label=r"TraPPE / lit. methane $T_c\approx 190$ K")
ax.set_xlabel(r"$\rho$ (kg m$^{-3}$)")
ax.set_ylabel("T (K)")
ax.set_title("VLCC: vapour–liquid coexistence curve")
ax.legend(loc="lower right", fontsize=8)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


## 5. Input–output discussion

1. Which single MDP line selects the coexistence point on the curve?
2. What would happen if you turned on an isotropic barostat on this slab?
3. Where does the methane force field live in this project — and how did Part 1 show that the interaction model affects phases?
4. Link to Sofia: why is a coexistence curve a **validation** observable for a methane model?  
   How does that story extend to **Coulomb** parameters in water/clay force fields, even though this UA methane has q = 0?
5. From your coexistence points (110–140 K), what classroom $T_c$ did the **scaling fit** give? TraPPE methane is $\approx 190\,\mathrm{K}$ and usually matches or slightly **over**estimates experiment — why is *our* estimate typically **lower**?
6. Geological hook (optional): methane is the fluid in natural gas and in **methane hydrate** research (e.g. South China Sea pilot production). We did **not** simulate a hydrate cage today — only bulk liquid–vapour coexistence.

---

**Tomorrow:** Mehdi Ghasemi — same GROMACS habits, molecular H₂–brine interfaces.  
**Day 3:** Sarah Stewart — clay–fluid systems.
